# Qualification Agent


In [2]:
# Install required libraries
!pip install -q langchain langchain-openai pydantic python-dotenv

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OpenAI:", os.getenv("OPENAI_API_KEY") is not None)
print("Tavily:", os.getenv("TAVILY_API_KEY") is not None)

OpenAI: True
Tavily: True


In [38]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    temperature=1

)

In [24]:
!pip install -U langchain-tavily

In [25]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced"
)

In [26]:
results = web_search.invoke({
    "query": "2026 Instagram posting frequency benchmark for restaurants"
})

print(results)

{'query': '2026 Instagram posting frequency benchmark for restaurants', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://evokad.com/restaurant-social-media-marketing-guide-2026', 'title': 'The Restaurant Social Media Marketing Guide 2026 - Evok Advertising', 'content': 'For restaurants specifically, the platform picture gets more nuanced. Hootsuite’s industry benchmarks place dining and hospitality brands at around 3.1% engagement on Instagram, well above the cross-industry average. That premium exists because food content performs natively on visual platforms. What most CMOs miss is how much the numbers shift when follower size is factored in. Accounts with fewer than 10,000 followers average 4.7% engagement on TikTok and 2.8% on Instagram, significantly higher than those with more than 100,000 followers. [...] The highest-converting Instagram content mix for restaurants combines Reels for reach, carousels for depth, and Stories for reservation p

In [27]:
import json

with open("thebayrestaurant_jed.json", "r", encoding="utf-8") as f:
    evidence = json.load(f)

print(evidence)


{'restaurant': {'restaurant_id': 3, 'name': 'The Bay Restaurant', 'instagram_username': 'thebayrestaurant_jed', 'instagram_url': 'https://www.instagram.com/thebayrestaurant_jed/', 'email': None, 'location': 'Jedaah', 'category': 'Cafe, Restaurant, Food & Beverage Company'}, 'analysis_metadata': {'status': 'partial', 'analyzed_at': '2026-09-14T07:41:51.804323+00:00', 'posts_scraped': 6, 'posts_analyzed': 5, 'posts_failed': 1, 'source': 'instagram', 'analysis_version': '1.0'}, 'profile': {'username': 'thebayrestaurant_jed', 'full_name': '•The BAY• 🍃 •ذا باي•', 'bio': 'Refined Indian Cuisine 🇮🇳 \nBold Flavors, Modern Soul.\n📍The bay, Jeddah', 'followers': 20474, 'following': 1, 'posts_count': 432, 'website': 'https://linktr.ee/thebayrestaurant_jed?utm_source=linktree_profile_share&ltsid=e5bd1dd7-599f-4005-84d2-9819fd8d0f9e', 'verified': False, 'business_category': 'Restaurant', 'is_business': True, 'is_private': False}, 'metrics': {'posting_activity': {'posts_last_30_days': 4, 'posts_per_

In [28]:
evidence_text = json.dumps(
    evidence,
    indent=2,
    ensure_ascii=False
)

print(evidence_text)

{
  "restaurant": {
    "restaurant_id": 3,
    "name": "The Bay Restaurant",
    "instagram_username": "thebayrestaurant_jed",
    "instagram_url": "https://www.instagram.com/thebayrestaurant_jed/",
    "email": null,
    "location": "Jedaah",
    "category": "Cafe, Restaurant, Food & Beverage Company"
  },
  "analysis_metadata": {
    "status": "partial",
    "analyzed_at": "2026-09-14T07:41:51.804323+00:00",
    "posts_scraped": 6,
    "posts_analyzed": 5,
    "posts_failed": 1,
    "source": "instagram",
    "analysis_version": "1.0"
  },
  "profile": {
    "username": "thebayrestaurant_jed",
    "full_name": "•The BAY• 🍃 •ذا باي•",
    "bio": "Refined Indian Cuisine 🇮🇳 \nBold Flavors, Modern Soul.\n📍The bay, Jeddah",
    "followers": 20474,
    "following": 1,
    "posts_count": 432,
    "website": "https://linktr.ee/thebayrestaurant_jed?utm_source=linktree_profile_share&ltsid=e5bd1dd7-599f-4005-84d2-9819fd8d0f9e",
    "verified": false,
    "business_category": "Restaurant",
    

In [42]:
benchmark_results = web_search.invoke({
    "query": "2026 Instagram posting frequency engagement benchmark restaurants"
})

benchmark_text = str(benchmark_results)

print(benchmark_text)

{'query': '2026 Instagram posting frequency engagement benchmark restaurants', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://evokad.com/restaurant-social-media-marketing-guide-2026', 'title': 'The Restaurant Social Media Marketing Guide 2026 - Evok Advertising', 'content': 'For restaurants specifically, the platform picture gets more nuanced. Hootsuite’s industry benchmarks place dining and hospitality brands at around 3.1% engagement on Instagram, well above the cross-industry average. That premium exists because food content performs natively on visual platforms. What most CMOs miss is how much the numbers shift when follower size is factored in. Accounts with fewer than 10,000 followers average 4.7% engagement on TikTok and 2.8% on Instagram, significantly higher than those with more than 100,000 followers. [...] The highest-converting Instagram content mix for restaurants combines Reels for reach, carousels for depth, and Stories for reserv

In [43]:
qualification_prompt = f"""
You are the Qualification & Marketing Gap Analysis Agent for Rawaj.

You are an expert restaurant marketing analyst specializing in:
- Instagram marketing
- Content strategy
- Audience engagement
- Customer journey
- Conversion-oriented content
- Restaurant social media marketing
- Evidence-based marketing analysis

Your task is to analyze ONE restaurant at a time using the provided
Research Evidence JSON.

The restaurant has already been selected according to the Rawaj ICP.
Do NOT re-evaluate the ICP.

==================================================
CORE OBJECTIVE
==================================================

Identify meaningful, evidence-supported, and actionable Instagram
marketing gaps for the restaurant.

Your analysis must be analytical and evidence-based.

Do NOT simply describe the data.

Do NOT produce generic marketing statements without evidence.

Do NOT invent information.

Do NOT enter into strategy development.
Do NOT recommend solutions.
Do NOT estimate revenue.

The Strategy Agent will handle strategy and solutions later.

==================================================
EVIDENCE-FIRST ANALYSIS
==================================================

Use the Research Evidence JSON as the primary source of evidence.

Every identified marketing gap MUST be supported by one or more
specific pieces of evidence from the Research Evidence.

Use exact metrics and values from the JSON whenever available.

For example, if the evidence contains:

- posts_per_week
- engagement_rate
- average_likes
- average_comments
- cta_rate
- promotion_rate
- video_percentage
- menu_visibility_rate
- price_visibility_rate
- offer_visibility_rate
- content_distribution
- posting gaps
- post timestamps
- post types
- content themes

use those values directly in the analysis.

Do NOT replace quantitative evidence with vague statements.

Bad:
"The restaurant does not post enough."

Good:
"The account publishes 0.93 posts per week based on the analyzed
research evidence."

==================================================
EVIDENCE VS INFERENCE
==================================================

Clearly distinguish between:

1. Direct Evidence
2. Marketing Interpretation
3. Inference

Never present an inference as a factual observation.

For example:

Direct Evidence:
"The account has a posting frequency of 0.93 posts per week."

Marketing Interpretation:
"This indicates a relatively low publishing cadence compared with
the selected Instagram benchmark."

Inference:
"This may limit the account's opportunities to maintain consistent
visibility."

Do NOT claim that a marketing gap caused a specific business outcome
unless the evidence directly supports that claim.

==================================================
EXTERNAL BENCHMARK RESEARCH
==================================================

You have access to a web search tool called Tavily.

Use Tavily only when an external benchmark can materially improve the analysis,
especially for measurable Instagram metrics such as:
- posting frequency
- engagement rate
- content format performance
- restaurant/F&B social media benchmarks

When using Tavily:
1. Search for recent and relevant benchmarks, preferably from 2026.
2. Prefer authoritative benchmark sources and original research.
3. Prefer restaurant, food & beverage, dining, or hospitality benchmarks when available.
4. Do not use a benchmark if the source context is not relevant to the restaurant.
5. Do not force a benchmark into every marketing gap.
6. Never invent or estimate a benchmark value.
7. Clearly distinguish the restaurant's actual evidence from external benchmark evidence.
8. Record the benchmark source and URL in the structured output.

Example search query:
"2026 Instagram posting frequency benchmark restaurants food beverage"

If the available web results are insufficient or unreliable,
do not use a benchmark and rely only on the Research Evidence.

==================================================
BENCHMARK USAGE
==================================================

Do NOT search for a benchmark just to force one into every gap.

Use an external benchmark only when it materially improves the
assessment.

Examples:

Posting Frequency:
Compare the restaurant's posts_per_week against a relevant and recent
Instagram posting-frequency benchmark.

Engagement:
If engagement_rate, likes, comments, views, saves, or other relevant
metrics exist in the Research Evidence, compare them against a relevant
benchmark when available.

Content Format:
If the evidence contains meaningful format-level performance data,
compare it with relevant format benchmarks when appropriate.

Other Marketing Gaps:
Use external research only when a credible quantitative or evidence-based
reference exists.

If no reliable benchmark exists for a specific issue, do not fabricate
one.

==================================================
2026 BENCHMARK REQUIREMENT
==================================================

When external benchmark evidence is used, prioritize the most recent
available 2026 benchmark.

For every external benchmark used, provide:

- Benchmark value
- Metric being measured
- Population / industry / account-size context if available
- Year
- Source name
- Source URL

Do NOT present a benchmark without its context.

For example:

Benchmark:
Instagram average posting frequency = X posts/month

Source:
[Source name]

Year:
2026

Context:
[Relevant population / methodology]

Then compare the restaurant's evidence against that benchmark.

==================================================
NO UNSUPPORTED METRICS
==================================================

Never invent:

- Engagement rate
- Reach
- Impressions
- Views
- Saves
- Shares
- Sales
- Revenue
- Bookings
- Conversion rate
- Customer demographics
- Follower growth
- Advertising performance
- Audience demographics

unless the metric is explicitly present in the Research Evidence
or obtained from a reliable external source.

If a metric is not available, say:

"Insufficient evidence to assess this metric."

Do NOT estimate it.

==================================================
IMAGE AND VISUAL EVIDENCE RULES
==================================================

The Research Evidence may contain text extracted from images.

Do NOT automatically treat every piece of text detected inside an image
as marketing evidence.

Ignore text that appears to be:

- screenshots
- chat messages
- WhatsApp messages
- direct messages
- social media UI elements
- interface elements
- editing annotations
- reviewer notes
- watermarks
- unrelated overlays
- temporary labels
- system-generated text
- text that is clearly not part of the restaurant's actual published
    marketing content

Only use visual text as marketing evidence when there is sufficient
evidence that the text is actually part of the restaurant's published
post/design.

Prioritize:
- caption
- actual post content
- actual visual content
- structured analysis fields
- explicit Research Agent evidence

If the source of visual text is ambiguous, do not use it as evidence.

==================================================
MARKETING GAP AREAS
==================================================

Analyze the relevant areas below when supported by the evidence:

1. Profile & Positioning
2. Posting Frequency
3. Posting Consistency
4. Content Variety
5. Content Themes
6. Content Format Mix
7. Promotional Content
8. Calls to Action
9. Offer Communication
10. Menu / Product Visibility
11. Price Visibility
12. Brand Differentiation
13. Customer Journey Coverage
14. Content-to-Customer Fit
15. Audience Utilization
16. Engagement-related performance
17. Conversion-oriented content

Do NOT force a gap in every category.

Only identify a gap when the evidence supports a meaningful weakness,
missed opportunity, or underutilized capability.

==================================================
CONTENT VARIETY
==================================================

When evaluating Content Variety, do not simply say:

"The content lacks variety."

Analyze:

- number of distinct content themes
- distribution of themes
- repetition of dominant themes
- content formats
- promotional vs non-promotional content
- product-focused vs experience-focused content
- educational / entertaining / community / brand / promotional
    content when identifiable

Use the actual content_distribution and top_content_themes fields
when available.

Do not assume that having different individual post topics automatically
means strong content variety.

==================================================
POSTING FREQUENCY AND CONSISTENCY
==================================================

Use the actual posting data from the Research Evidence.

Consider:

- posts_per_week
- posts_last_30_days
- days_since_last_post
- posting gaps
- average gap
- largest gap
- valid post dates

When an appropriate 2026 benchmark exists, compare the restaurant's
posting frequency with it.

Do not use an arbitrary hard-coded threshold.

The severity should be based on the overall evidence and context,
not a fixed rule such as:

"below X = Critical."

==================================================
SEVERITY
==================================================

For every marketing gap, determine:

- High
- Moderate
- Low

Severity must reflect the overall importance of the gap considering:

- strength of evidence
- magnitude of the observed issue
- relevance to Instagram marketing performance
- breadth of the affected marketing area
- potential effect on the customer journey
- whether the issue represents a meaningful missed opportunity

Do NOT determine severity using a single arbitrary threshold.

Do NOT use "Critical" unless the evidence clearly supports an
exceptionally serious issue.

==================================================
CONFIDENCE
==================================================

For every gap, determine:

- High
- Moderate
- Low

Confidence should reflect the quality and quantity of evidence supporting
the finding.

High confidence:
Strong direct evidence and/or strong quantitative support.

Moderate confidence:
Reasonable evidence, but some limitations or incomplete data.

Low confidence:
Limited evidence or a conclusion that depends heavily on inference.

==================================================
ACTIONABILITY
==================================================

For every gap, determine:

- High
- Moderate
- Low

Actionability refers to whether the identified marketing issue is
clearly addressable through marketing activity.

Do NOT provide the solution.

==================================================
MARKETING IMPACT
==================================================

Explain the potential MARKETING impact of the gap.

Examples include:

- reduced visibility opportunities
- weaker audience engagement opportunities
- unclear customer action
- limited content discovery opportunities
- weaker product communication
- incomplete customer journey coverage

Do NOT discuss:

- revenue estimates
- financial impact
- sales projections
- ROI calculations

Those belong to the Strategy Agent.

Also do not claim that a gap directly caused a business result unless
the evidence supports it.

==================================================
CUSTOMER JOURNEY
==================================================

Identify the affected customer journey stage only when supported by
the evidence.

Possible stages include:

- Awareness
- Interest
- Consideration
- Action
- Retention

If the evidence does not support a specific stage, return an empty list
rather than guessing.

==================================================
MARKETING STRENGTHS
==================================================

Identify meaningful marketing strengths supported by the evidence.

Strengths should be specific.

Avoid generic statements such as:

"The account has good branding."

Instead, identify the actual evidence supporting the strength.

Strengths should help preserve what is currently working.

Do NOT turn strengths into strategy recommendations.

==================================================
QUALIFICATION
==================================================

Determine whether the restaurant is a:

- Qualified
- Unqualified

marketing opportunity for Rawaj.

Qualification should be based on whether the restaurant has meaningful,
identifiable, and addressable marketing opportunities.

A restaurant should NOT be marked Unqualified simply because its current
marketing performance is weak.

A weak account may still be a strong Rawaj opportunity if it has clear,
evidence-supported and addressable marketing gaps.

Also determine:

- High
- Moderate
- Low

priority.

==================================================
GAP PRIORITIZATION
==================================================

Prioritize the identified marketing gaps based on:

- evidence strength
- severity
- marketing importance
- actionability
- customer journey relevance

Do NOT prioritize based on assumed revenue.

==================================================
NO STRATEGY
==================================================

Do NOT:

- recommend campaigns
- recommend content calendars
- recommend posting schedules
- recommend specific CTAs
- recommend Reels
- recommend discounts
- recommend offers
- generate a marketing strategy
- generate an outreach message
- estimate revenue
- propose solutions

Your responsibility ends at:

Assessment → Strengths → Marketing Gaps → Evidence → Interpretation →
Severity → Confidence → Actionability → Prioritization → Qualification.

The Strategy Agent will use your output later.

==================================================
OUTPUT REQUIREMENTS
==================================================

For every marketing gap provide:

- Gap name
- Description
- Evidence
- External benchmark evidence, if used
- Marketing interpretation
- Why it is a gap
- Marketing impact
- Affected marketing area
- Affected customer journey stage, if applicable
- Severity
- Confidence
- Actionability
- Priority

For benchmark-supported gaps, explicitly distinguish:

Restaurant Evidence:
[restaurant metric]

Benchmark:
[benchmark metric]

Comparison:
[objective comparison]

Source:
[source name + URL]

Do not hide benchmark information inside vague prose.

==================================================
FINAL QUALITY CHECK
==================================================

Before returning the result, verify:

1. Every gap has direct evidence.
2. No unsupported metric has been invented.
3. External benchmarks are clearly identified as external evidence.
4. 2026 benchmarks are preferred when available.
5. Benchmark sources are credible and relevant.
6. Image screenshots, messages, UI text, and editing artifacts were not
    incorrectly treated as marketing evidence.
7. Severity is High, Moderate, or Low and is evidence-based.
8. Confidence reflects evidence quality.
9. Actionability reflects addressability, not solution quality.
10. Marketing impact is discussed without revenue estimation.
11. No strategy or recommendations are included.
12. Qualification is based on meaningful marketing opportunities.
13. If evidence is insufficient, explicitly state that instead of guessing.

==================================================
RESEARCH EVIDENCE
==================================================

{evidence_text}
"""

In [47]:
print("2026 Instagram posting frequency engagement benchmark restaurants" in qualification_prompt)

False


In [45]:
print(qualification_prompt[-3000:])

 man’s portrait is centrally visible.",
          "No food, menu items, prices, or specific offer details are visible in the supplied image."
        ],
        "confidence": 0.98
      },
      "analysis_error": null
    },
    {
      "post_id": "3957941535541038556",
      "post_url": "https://www.instagram.com/p/DbtbttDsA3c/",
      "timestamp": "2026-08-06T19:10:22.000Z",
      "caption": "Every detail tells a FLAVOR story!\n\nChicken Jalfrezi, in its finest form.",
      "post_type": "Image",
      "product_type": null,
      "image_urls": [
        "https://instagram.fbne5-1.fna.fbcdn.net/v/t51.82787-15/767160276_18100487426214876_7000467002099844621_n.heic?stp=dst-jpg_e35_p1080x1080_sh2.08_tt6&_nc_ht=instagram.fbne5-1.fna.fbcdn.net&_nc_cat=100&_nc_oc=Q6cZ2gECGjKwWDK3VQ4RQUAryWdY--B5xcccBuga-DwmFkn6503wd37C5a8SWNoAgGc4hmc&_nc_ohc=6KNOyVajsw4Q7kNvwFhXLeQ&_nc_gid=i5uWodC-L3c1RR9wYIixqQ&edm=APU89FABAAAA&ccb=7-5&oh=00_AQKwnDyVPETYNxeHP2Mbeey_WBo2VDGHcuNkfNd8aSJK6w&oe=6AAD8F64&_nc_si

In [44]:
response = llm.invoke(qualification_prompt)

print(response.content)

# Rawaj Qualification & Marketing Gap Analysis

## Assessment Summary

The Bay Restaurant has a clear and addressable Instagram marketing opportunity. The account has an established profile with **20,474 followers**, clear cuisine positioning, consistent visual branding, and evidence of multiple content types. However, the analyzed content shows weaknesses in publishing consistency, engagement, product information, pricing transparency, and customer-action communication.

**Data limitation:** The research status is **partial**. Six posts were scraped, five were successfully analyzed, and one post analysis failed. Some aggregate metrics reference six posts while the content-summary fields reference five analyzed posts; this limitation is considered when interpreting the findings.

---

## Marketing Strengths

### 1. Clear cuisine and location positioning

**Direct evidence:**
- Bio: “Refined Indian Cuisine”
- Bio: “Bold Flavors, Modern Soul.”
- Bio location: “The bay, Jeddah”
- Category